# 12.17 - Multi-Query RAG

**Phase:** 12 - LangChain

**Status:** VERIFIED

---

## 1. What Are We Solving?

A single query may miss relevant documents. Multi-query RAG generates multiple perspectives of the same question.

## 2. Why Does This Matter?

Different phrasings retrieve different documents. More queries = better coverage.

## 3. Prerequisites

- 12.07: RAG with LangChain
- 12.16: RAG evaluation

## 4. Learning Objectives

- Generate multiple query perspectives
- Aggregate results from multiple queries
- Improve retrieval recall

## 5. Mental Model

Original query -> generate N variations -> retrieve for each -> merge results -> answer.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
print("Libraries loaded.")

Libraries loaded.


## 6. Multi-Query Generator

In [2]:
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.3)

query_prompt = ChatPromptTemplate.from_template(
    "You are an AI assistant. Generate {n} different versions of the following user question to retrieve relevant documents.\n\nOriginal question: {question}\n\nOutput each variation on a new line, numbered 1-{n}:"
)

def generate_queries(question, n=3):
    chain = query_prompt | llm | StrOutputParser()
    result = chain.invoke({"question": question, "n": n})
    queries = [line.split(".", 1)[1].strip() for line in result.split("\n") if "." in line]
    return queries[:n]

queries = generate_queries("What is machine learning?")
print("Generated queries:")
for i, q in enumerate(queries, 1):
    print("  " + str(i) + ". " + q)

Generated queries:
  1. Define machine learning and explain its core principles.
  2. What are the key concepts and applications of machine learning?
  3. Provide an overview of what machine learning is and how it works.


## 7. Multi-Query Retrieval

In [3]:
docs = [
    Document(page_content="Machine learning is a subset of AI.", metadata={"source": "ml.txt"}),
    Document(page_content="Deep learning uses neural networks.", metadata={"source": "dl.txt"}),
    Document(page_content="Python is used for ML development.", metadata={"source": "python.txt"}),
    Document(page_content="Supervised learning uses labeled data.", metadata={"source": "supervised.txt"}),
]

def multi_query_retrieve(question, docs, n=3):
    queries = generate_queries(question, n=n)
    all_results = []
    seen = set()
    
    for q in queries:
        query_words = set(q.lower().split())
        for doc in docs:
            doc_words = set(doc.page_content.lower().split())
            overlap = len(query_words & doc_words)
            if doc.metadata["source"] not in seen:
                all_results.append((doc, overlap))
                seen.add(doc.metadata["source"])
    
    all_results.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in all_results[:4]]

results = multi_query_retrieve("What is machine learning?", docs)
print("Retrieved " + str(len(results)) + " documents")
for d in results:
    print("  -", d.metadata["source"], ":", d.page_content[:60])

Retrieved 4 documents
  - ml.txt : Machine learning is a subset of AI.
  - dl.txt : Deep learning uses neural networks.
  - supervised.txt : Supervised learning uses labeled data.
  - python.txt : Python is used for ML development.


## 8. Answer with Multi-Query

In [4]:
rag_prompt = ChatPromptTemplate.from_template(
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)
rag_chain = rag_prompt | llm | StrOutputParser()

def multi_query_rag(question, docs):
    retrieved = multi_query_retrieve(question, docs)
    context = "\n".join(d.page_content for d in retrieved)
    answer = rag_chain.invoke({"context": context, "question": question})
    return answer

answer = multi_query_rag("What is machine learning?", docs)
print("Answer:", answer[:300])

Answer: Based on the provided context, machine learning is a subset of AI.


## 9. Common Mistakes

1. Too many query variations (slow)
2. Queries too similar (no benefit)
3. Not deduplicating results
4. Ignoring query quality

## 10. Coding Exercises

### Exercise 1: Compare Single vs Multi-Query
Compare retrieval quality.

### Exercise 2: Adaptive Multi-Query
Dynamically decide how many queries to generate.

In [5]:
# EXERCISE 1
print("Exercise: Compare single vs multi-query retrieval.")

Exercise: Compare single vs multi-query retrieval.


In [6]:
# EXERCISE 2
print("Exercise: Build adaptive multi-query system.")

Exercise: Build adaptive multi-query system.


## 11. Closed-Book Recall

1. Why use multiple queries?
2. How do you merge results?
3. When is multi-query most helpful?

## 12. Summary

Multi-query RAG generates multiple perspectives. Deduplicate results. Improve recall by covering more angles.

## Verification Status
```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: [langchain, langchain-groq]
OUTPUTS: PASS
LAST VERIFIED: 2026-08-30
```